In [ ]:
import os
import numpy as np
import pandas as pd
import warnings
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

os.environ['PYTHONHASHSEED'] = str(1)
np.random.seed(1)
warnings.filterwarnings("ignore")

MODEL_FILES = {
    'LightGBM': 'LightGBM_Evaluation_Output.xlsx',
    'CatBoost': 'CatBoost_Evaluation_Output.xlsx',
    'GBoost': 'GradientBoost_Evaluation_Output.xlsx',
    'XGBoost':  'XGBoost_Evaluation_Output.xlsx'
}

def load_and_align_data(file_map):
    metrics_list = []
    preds_list = []
    
    for model_name, filepath in file_map.items():
        if not os.path.exists(filepath):
            raise FileNotFoundError(f"File not found: {filepath}")
            
        df_m = pd.read_excel(filepath, sheet_name='Metrics_Score_Table')
        df_m.columns = df_m.columns.str.strip()
        df_m['Model'] = model_name
        metrics_list.append(df_m)
        
        df_p = pd.read_excel(filepath, sheet_name='Predictions_Target_Sample')
        df_p.columns = df_p.columns.str.strip()
        
        pred_col_candidates = [c for c in df_p.columns if 'Pred' in c]
        if not pred_col_candidates:
            raise KeyError(f"No prediction column found in {model_name}")
        
        target_pred_col = pred_col_candidates[0]
        df_p = df_p.rename(columns={target_pred_col: f'Pred_{model_name}'})
        
        if 'Sample_Index' not in df_p.columns:
            df_p['Sample_Index'] = df_p.index
            
        preds_list.append(df_p[['Sample_Index', 'Actual_Value', f'Pred_{model_name}']])

    metrics_df = pd.concat(metrics_list, ignore_index=True).set_index('Model')
    
    fusion_df = preds_list[0]
    for i in range(1, len(preds_list)):
        fusion_df = pd.merge(
            fusion_df, 
            preds_list[i][['Sample_Index', f'Pred_{list(file_map.keys())[i]}']], 
            on='Sample_Index', 
            how='inner'
        )
        
    return metrics_df, fusion_df

def entropy_topsis_weights(metrics_df):
    """
    Implements Entropy-TOPSIS algorithm for model weighting.
    
    Principles:
    1. Standardization (Min-Max):
       Benefit (R2): x_ij = (v_ij - min(v_j)) / (max(v_j) - min(v_j))
       Cost (RMSE/MAE): x_ij = (max(v_j) - v_ij) / (max(v_j) - min(v_j))
       
    2. Entropy Weight (W_j):
       H_j = -k * sum(p_ij * ln(p_ij)), where k = 1/ln(n)
       w_j = (1 - H_j) / sum(1 - H_k)
       
    3. TOPSIS Score (C_i):
       Z_ij = w_j * x_ij
       D_i+ = sqrt(sum((Z_ij - Z_j+)^2))
       D_i- = sqrt(sum((Z_ij - Z_j-)^2))
       C_i = D_i- / (D_i+ + D_i-)
    """
    data = metrics_df[['MSE', 'RMSE', 'MAE', 'R2']].copy()
    directions = {'MSE': -1, 'RMSE': -1, 'MAE': -1, 'R2': 1}
    
    # 1. Standardization
    norm_df = pd.DataFrame(index=data.index)
    epsilon = 1e-9
    
    for col in data.columns:
        vals = data[col].values
        if directions[col] == 1:
            norm_df[col] = (vals - vals.min()) / (vals.max() - vals.min() + epsilon)
        else:
            norm_df[col] = (vals.max() - vals) / (vals.max() - vals.min() + epsilon)
            
    # 2. Entropy Method
    P = norm_df.div(norm_df.sum(axis=0), axis=1) + epsilon
    k = 1.0 / np.log(len(data))
    E = -k * (P * np.log(P)).sum(axis=0)
    D = 1 - E
    obj_weights = D / D.sum()
    
    # 3. TOPSIS
    Z = norm_df * obj_weights
    Z_plus = Z.max()
    Z_minus = Z.min()
    
    D_plus = np.sqrt(((Z - Z_plus) ** 2).sum(axis=1))
    D_minus = np.sqrt(((Z - Z_minus) ** 2).sum(axis=1))
    
    C = D_minus / (D_plus + D_minus)
    final_weights = C / C.sum()
    
    return final_weights

def fuse_predictions(fusion_df, weights):
    fusion_df['Pred_ETFM'] = 0.0
    for model, weight in weights.items():
        fusion_df['Pred_ETFM'] += fusion_df[f'Pred_{model}'] * weight
    return fusion_df

def export_results(metrics_df, fusion_df, weights, output_file):
    y_true = fusion_df['Actual_Value']
    y_pred = fusion_df['Pred_ETFM']
    
    etfm_score = pd.DataFrame({
        'Model': ['ETFM_Ensemble'],
        'MSE': [mean_squared_error(y_true, y_pred)],
        'RMSE': [np.sqrt(mean_squared_error(y_true, y_pred))],
        'MAE': [mean_absolute_error(y_true, y_pred)],
        'R2': [r2_score(y_true, y_pred)]
    }).set_index('Model')
    
    final_metrics = pd.concat([metrics_df[['MSE', 'RMSE', 'MAE', 'R2']], etfm_score])
    
    with pd.ExcelWriter(output_file) as writer:
        weights.to_frame('Weight').to_excel(writer, sheet_name='Weights')
        final_metrics.to_excel(writer, sheet_name='Metrics_Comparison')
        fusion_df.to_excel(writer, sheet_name='Detailed_Predictions', index=False)
        
    print(f"Final R2: {r2_score(y_true, y_pred):.4f}")
    print(f"Results saved to {output_file}")

def main():
    try:
        print("Loading data...")
        metrics_df, fusion_df = load_and_align_data(MODEL_FILES)
        
        print("Calculating Entropy-TOPSIS weights...")
        weights = entropy_topsis_weights(metrics_df)
        print("Weights:\n", weights)
        
        print("Fusing models...")
        final_df = fuse_predictions(fusion_df, weights)
        
        export_results(metrics_df, final_df, weights, 'ETFM_Final_Results.xlsx')
        
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    main()